# Capstone: An Earnings-Call NLP Pipeline

formalNLP, Topic 29. This notebook walks through the topic's claims by calling the reference implementation,
`capstone_earnings_call_nlp.py`, which owns every number on the page and in the lab. Nothing here redefines the
mathematics.

Four published stages read an earnings call: entities (Topic 22's tagger, linked by Topic 23's string match), relations
(Topic 23's MIL reader), tone (Topic 18's dictionary, Topic 17's LR, or Topic 19's aspect reader), and aspects (a term
matcher with Topic 19's bigram reader). Their output is a graph: the facts the calls state, and for every call its
subject, tone, and aspect polarities. The page asks three questions of the composition: which stage costs what (Thm 1),
what a stage should hand the next one (Thm 2), and whether the KB's own constraints can repair the output (Thm 3).

In [ ]:
import itertools
import sys
from pathlib import Path

sys.path.insert(0, str(Path("notebooks/capstone-earnings-call-nlp").resolve()))
import numpy as np
import capstone_earnings_call_nlp as cap

## 1. The pipeline and its oracles (Def 1)

The test documents are Topic 22's two runs of `generate_corpus(200, 1)`: one on the training roster ("seen") and one
on the six held-out companies ("unseen"). We read their earnings calls. A stage's oracle replaces its prediction by
the gold output; every oracle reproduces its prerequisite's own gold reading.

In [ ]:
for n in cap.TESTS:
    print(f"{n:7s} calls {len(cap.calls(n))}  companies {len(cap.companies(n))}  gold facts {len(cap.gold_facts(n))}"
          f"  gold triples {len(cap.gold_graph(n))}")
cap.test_stage_e_is_topic23s_and_subjects_are_the_covered_company()
cap.test_gold_side_is_topic19s_and_topic27s()
print("stage E is Topic 23's; the gold sides are Topic 19's tone_sentences and Topic 27's doc_facts")

## 2. Two-sided bounds (Prop 1)

Let $A$ be the event that a fact's two arguments are found and linked, and $B$ the event that the reader recovers it from
gold mentions. Bonferroni gives $P(A \cap B) \ge r_E + r_R - 1$, and $P(A \cap B) \le \min(r_E, r_R)$. formalRAG's product
law is exact under independence. End to end equals $P(A \cap B)$ only when the reader reads found arguments as it reads
gold ones.

In [ ]:
for view in cap.VIEWS:
    b = cap.recall_bounds(view)
    print(f"{view:7s} r_E {b['r_E']:.3f} r_R {b['r_R']:.3f} | Bonferroni {b['bonferroni']:.3f}  product {b['product']:.3f}"
          f"  P(A∩B) {b['intersection']:.3f}  end-to-end {b['end']:.3f}  min {b['min']:.3f}")
cap.test_prop1_two_sided_bounds_and_modularity(); print("Prop 1 holds; the seen calls are not modular")

## 3. The oracle game (Def 2, Thm 1, Cor 1, Prop 2)

$v(S)$ is the call graph's micro F1 with the stages in $S$ replaced by their oracles, minus $v(\varnothing)$. A ceiling
analysis adds the oracles in one order. Its share for stage $i$ is the sum of the Harsanyi dividends $d(T)$ of the
coalitions that $i$ completes. The Shapley value splits each dividend equally among its members.

In [ ]:
v = cap.game("both", "LR")
d = cap.dividends(v)
print("Shapley:", {i: round(x, 4) for i, x in cap.shapley(v).items()})
print("dividends of two or more stages:", {cap._key(T): round(x, 4) for T, x in d.items() if len(T) >= 2 and abs(x) > 1e-12})
for o in (("E", "R", "T", "A"), ("R", "E", "T", "A"), ("T", "R", "E", "A")):
    print("order", "".join(o), {i: round(x, 4) for i, x in cap.sequential(v, o).items()})
cap.test_thm1_ceiling_analysis_and_the_shapley_value()
cap.test_thm1_on_the_corpus_entities_and_relations_are_complements(); print("Thm 1 on random games and on the corpus")

The aspect stage is a null player. The matcher finds exactly the gold spans, the bigram reader reads every one right,
and so its oracle changes nothing (Cor 1). The error game counts errors removed. Stages that write disjoint triples add
in it exactly, while micro F1 couples them through its denominator (Prop 2).

In [ ]:
cap.test_cor1_the_aspect_stage_is_a_null_player()
e = cap.game("both", "LR", "errors")
print("error game Shapley:", {i: x for i, x in cap.shapley(e).items()})
print("error-game dividends:", {cap._key(T): x for T, x in cap.dividends(e).items() if len(T) >= 2 and x})
cap.test_prop2_errors_add_over_disjoint_triples_f1_does_not(); print("Cor 1 and Prop 2 hold")

In [ ]:
for view in cap.VIEWS:
    for reader in cap.READERS:
        phi = cap.shapley(cap.game(view, reader))
        print(f"{view:7s} {reader:10s}", "  ".join(f"{cap.STAGE_NAMES[i]} {phi[i]:.4f}" for i in cap.STAGES))

## 4. Handoffs (Def 3, Thm 2)

A Bayes downstream does at least as well with the posterior as with its argmax, because the argmax is a garbling of the
posterior (Thm 2a). A fixed downstream rule can still lose (Thm 2b). On the corpus, Topic 22's span marginals and
samples raise fact AP where entity errors are common. They also lower F1 at the hard pipeline's threshold, which a
threshold tuned on the same run's filings recovers.

In [ ]:
cap.test_thm2a_the_argmax_is_a_garbling()
print("Thm 2b:", cap.counterexample()); cap.test_thm2b_a_fixed_rule_can_lose()
cap.test_ffbs_samples_the_constrained_chain(); print("forward filtering, backward sampling matches the chain")
for n in cap.TESTS:
    t = cap.handoff_table(n)
    for how in cap.HANDOFFS:
        h = t[how]
        print(f"{n:7s} {how:9s} F1@0.5 {h['at']['f1']:.3f}  AP {h['ap']:.3f}  F*/2 {h['half']:.3f} → F1 {h['at_half']['f1']:.3f}"
              f"  tuned on filings τ={h['tuned']:g} → F1 {h['at_tuned']['f1']:.3f}")
cap.test_handoffs_on_the_corpus()

## 5. Joint inference over the KB (Def 4, Thm 3, Prop 3)

We merge the inverse and symmetric facts into single variables. The functional rows then give a totally unimodular
matrix, so the LP relaxation is the MAP. Composition rules break total unimodularity, though on this corpus the LP
stays integral anyway.

In [ ]:
cap.test_thm3_functional_rows_are_totally_unimodular()
cap.test_prop3_composition_rules_break_total_unimodularity()
x, val = cap.solve_kb({"w": cap.PROP3_W, "A": cap.sparse.csr_matrix(cap.PROP3_A), "b": cap.PROP3_B}, False)
print("Prop 3 LP point", x.round(3), "value", round(val, 3), "| best integer", cap.brute_ilp(cap.PROP3_W, cap.PROP3_A, cap.PROP3_B))
for n in cap.TESTS:
    t = cap.kb_table(n, False)
    for tau in (0.05, 0.1, 0.5, 0.8):
        k = t["tau"].index(tau)
        print(f"{n:7s} τ={tau:<4g}", "  ".join(f"{m} {t[m][k]['f1']:.3f}" for m in ("threshold", "merged", "functional", "rules")))
cap.test_joint_inference_on_the_corpus()

The unseen calls miss six facts. Four of them are in Topic 24's completion KB, extracted from the same companies'
filings, and the other two are their inverses. RotatE ranks all six first.

In [ ]:
c = cap.completion_ranks()
for f, inside, r in zip(c["missed"], c["in_filings"], np.array(c["ranks"]).reshape(-1, 2)):
    print(f, "in filings' KB" if inside else "inverse of a filed fact", "ranks", r)
cap.test_filings_fill_what_the_calls_missed()

## 6. How sure are the shares? (Prop 4) And the tone readers compared

The Shapley value is a linear map of $v$, and $v$ is a ratio of pooled counts. Resampling companies therefore gives every
share's bootstrap distribution in one matrix product.

In [ ]:
for view in cap.VIEWS:
    b = cap.share_bootstrap(view, "LR")
    print(f"{view:7s} 90% intervals", "  ".join(f"{cap.STAGE_NAMES[i]} [{lo:.4f}, {hi:.4f}]"
                                                 for i, lo, hi in zip(cap.STAGES, b["lo"], b["hi"])), f"  P(E > R) {b['p_E_gt_R']:.3f}")
cap.test_prop4_shapley_is_linear_so_one_bootstrap_serves_every_share()
for (a, b_), x in cap.reader_tests().items():
    print(f"{a} vs {b_}: {x['acc_a']:.3f} vs {x['acc_b']:.3f}  discordant {x['n10']}/{x['n01']}  McNemar p {x['mcnemar']:.4f}"
          f"  cluster AR p {x['cluster_ar']:.4f}")
cap.test_tone_readers_compared()

The topic stage folds each call into Topic 25's fit. Each generator run invents its own pseudo-words, so on seed-1
text they carry seed 0's sectors. Folding in only the 80 seeded nouns repairs it.

In [ ]:
for n in cap.TESTS:
    print(f"{n:7s} sector accuracy: all words {cap.topic_stage(n, False):.3f}   fixed nouns {cap.topic_stage(n, True):.3f}")
cap.test_topic_stage_needs_the_fixed_nouns()

## The full harness

Every claim above, plus the payload mirror, in one run.

In [ ]:
assert cap._run_harness() == 0